# Chest X-Ray Multi-Label Diagnosis — DenseNet-121 fine-tuned in PyTorch

Fine-tunes an ImageNet-pretrained **DenseNet-121** on the **NIH ChestX-ray14** dataset to detect 14 thoracic conditions from a single frontal X-ray, then explains predictions with **Grad-CAM**.

**What this notebook does**
1. Builds 14 binary labels from `Data_Entry_2017.csv`
2. Uses the official NIH test split and a **patient-level** train/val split (no patient appears in two splits)
3. Caches resized images in RAM for fast training
4. Trains with **class-weighted BCE loss** (handles rare classes like Hernia), mixed precision, OneCycle LR schedule and early stopping on validation macro-AUC
5. Reports per-class **ROC-AUC** on held-out test patients
6. Generates **Grad-CAM** heatmaps

**Kaggle setup:** *Add Input* → search **"NIH Chest X-rays"** (`nih-chest-xrays/data`) · *Accelerator* → **GPU T4** or **P100** · *Internet* → **On** (to download ImageNet weights).

Run once with `SMOKE_TEST = True` (≈3 min) to check everything works, then set it to `False` for the real run.

In [ ]:
# ---------------- CONFIG ----------------
SMOKE_TEST = False      # True = tiny quick run to check the pipeline end-to-end

N_TRAIN   = 30000       # training images (max ~78k). More = better AUC, longer run
N_VAL     = 5000
N_TEST    = 10000       # sampled from the official NIH test split (max ~25.6k)
CACHE_SIZE = 256        # images are resized once to this and kept in RAM
IMG_SIZE  = 224         # model input size
BATCH_SIZE = 32
EPOCHS    = 8
LR_BACKBONE = 1e-4
LR_HEAD     = 1e-3
WEIGHT_DECAY = 1e-5
POS_WEIGHT_CAP = 50.0   # caps neg/pos weight so ultra-rare classes don't destabilise training
PATIENCE  = 3           # early stopping on val macro-AUC
NUM_WORKERS = 4
SEED = 42
OUT_DIR = "/kaggle/working/outputs"

if SMOKE_TEST:
    N_TRAIN, N_VAL, N_TEST, EPOCHS = 512, 256, 256, 1

LABELS = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule', 'Pneumonia',
          'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia']

In [ ]:
import os, glob, json, time, random
from multiprocessing import Pool

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms as T
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import GroupShuffleSplit

os.makedirs(OUT_DIR, exist_ok=True)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| device:", DEVICE,
      torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

## 1. Load metadata and build labels

In [ ]:
csv_hits = glob.glob("/kaggle/input/**/Data_Entry_2017.csv", recursive=True)
assert csv_hits, "Data_Entry_2017.csv not found - add the 'NIH Chest X-rays' dataset as input"
DATA_DIR = os.path.dirname(csv_hits[0])
print("Data dir:", DATA_DIR)

df = pd.read_csv(csv_hits[0])
df = df.rename(columns={"Image Index": "image", "Finding Labels": "findings", "Patient ID": "patient_id"})

# map every image file name -> full path (images are spread over images_001 ... images_012)
paths = glob.glob(os.path.join(DATA_DIR, "images*", "**", "*.png"), recursive=True)
path_map = {os.path.basename(p): p for p in paths}
df["path"] = df["image"].map(path_map)
print(f"{len(df):,} rows | {df['path'].notna().sum():,} images found | {df['patient_id'].nunique():,} patients")
df = df[df["path"].notna()].reset_index(drop=True)

for lab in LABELS:
    df[lab] = df["findings"].str.split("|").apply(lambda f: int(lab in f))

df[LABELS].mean().sort_values().plot.barh(figsize=(7, 5), title="Label prevalence (full dataset)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/label_prevalence.png", dpi=120); plt.show()

## 2. Patient-level splits (no leakage)

In [ ]:
def read_list(name):
    with open(os.path.join(DATA_DIR, name)) as f:
        return set(x.strip() for x in f if x.strip())

test_names = read_list("test_list.txt")          # official NIH patient-wise test split
trainval = df[~df["image"].isin(test_names)]
test_all = df[df["image"].isin(test_names)]

gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED)
tr_idx, va_idx = next(gss.split(trainval, groups=trainval["patient_id"]))
train_all, val_all = trainval.iloc[tr_idx], trainval.iloc[va_idx]

train_df = train_all.sample(min(N_TRAIN, len(train_all)), random_state=SEED).reset_index(drop=True)
val_df   = val_all.sample(min(N_VAL, len(val_all)), random_state=SEED).reset_index(drop=True)
test_df  = test_all.sample(min(N_TEST, len(test_all)), random_state=SEED).reset_index(drop=True)

# leakage check: no patient in more than one split
p_tr, p_va, p_te = set(train_all.patient_id), set(val_all.patient_id), set(test_all.patient_id)
assert not (p_tr & p_va) and not (p_tr & p_te) and not (p_va & p_te), "patient leakage!"
print(f"train {len(train_df):,} | val {len(val_df):,} | test {len(test_df):,}  -> no patient overlap")

## 3. Cache resized images in RAM

In [ ]:
def load_resized(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    return cv2.resize(img, (CACHE_SIZE, CACHE_SIZE), interpolation=cv2.INTER_AREA)

def cache_images(frame, name):
    t0 = time.time()
    with Pool(os.cpu_count()) as pool:
        imgs = pool.map(load_resized, frame["path"].tolist(), chunksize=64)
    arr = np.stack(imgs).astype(np.uint8)
    print(f"{name}: cached {arr.shape} in {time.time() - t0:.0f}s ({arr.nbytes / 1e9:.2f} GB)")
    return arr

X_train = cache_images(train_df, "train")
X_val   = cache_images(val_df, "val")
X_test  = cache_images(test_df, "test")
Y_train, Y_val, Y_test = (d[LABELS].values.astype(np.float32) for d in (train_df, val_df, test_df))

## 4. Datasets and augmentation

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.80, 1.0), ratio=(0.9, 1.1), antialias=True),
    T.RandomRotation(7),
    T.ColorJitter(brightness=0.15, contrast=0.15),
    T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([T.Resize(IMG_SIZE, antialias=True), T.Normalize(MEAN, STD)])
# no horizontal flip: anatomy is not left-right symmetric (heart sits on the left)

class XrayDataset(Dataset):
    def __init__(self, images, labels, tf):
        self.images, self.labels, self.tf = images, labels, tf
    def __len__(self):
        return len(self.images)
    def __getitem__(self, i):
        x = torch.from_numpy(self.images[i]).float().div(255.).unsqueeze(0).repeat(3, 1, 1)
        return self.tf(x), torch.from_numpy(self.labels[i])

loader_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True)
train_dl = DataLoader(XrayDataset(X_train, Y_train, train_tf), shuffle=True, drop_last=True, **loader_kw)
val_dl   = DataLoader(XrayDataset(X_val, Y_val, eval_tf), shuffle=False, **loader_kw)
test_dl  = DataLoader(XrayDataset(X_test, Y_test, eval_tf), shuffle=False, **loader_kw)

xb, yb = next(iter(train_dl))
print("batch:", xb.shape, yb.shape)

## 5. Model — DenseNet-121 (ImageNet weights) with a 14-way sigmoid head

In [ ]:
class ChestXrayNet(nn.Module):
    def __init__(self, n_classes=len(LABELS)):
        super().__init__()
        base = torchvision.models.densenet121(weights=torchvision.models.DenseNet121_Weights.IMAGENET1K_V1)
        self.features = base.features                     # conv backbone (Grad-CAM hooks here)
        self.classifier = nn.Linear(base.classifier.in_features, n_classes)
    def forward(self, x):
        f = F.relu(self.features(x))                      # not in-place, so Grad-CAM gradients stay valid
        f = F.adaptive_avg_pool2d(f, 1).flatten(1)
        return self.classifier(f)                         # logits; sigmoid applied at inference

model = ChestXrayNet().to(DEVICE)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")

## 6. Class-weighted loss

Most labels are rare (Hernia ≈ 0.2%). Plain BCE lets the model ignore positives, so each class's positive term is weighted by `neg/pos`, capped at `POS_WEIGHT_CAP`.

In [ ]:
pos = Y_train.sum(0)
neg = len(Y_train) - pos
pos_weight = np.clip(neg / np.maximum(pos, 1), 1.0, POS_WEIGHT_CAP)
print(pd.DataFrame({"positives": pos.astype(int), "pos_weight": pos_weight.round(1)}, index=LABELS))
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, dtype=torch.float32, device=DEVICE))

## 7. Train

In [ ]:
optimizer = torch.optim.AdamW([
    {"params": model.features.parameters(), "lr": LR_BACKBONE},
    {"params": model.classifier.parameters(), "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=[LR_BACKBONE, LR_HEAD], total_steps=EPOCHS * len(train_dl), pct_start=0.1)
use_amp = DEVICE.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

@torch.no_grad()
def predict(dl):
    model.eval()
    probs, ys = [], []
    for x, y in dl:
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
            logits = model(x.to(DEVICE, non_blocking=True))
        probs.append(torch.sigmoid(logits.float()).cpu().numpy()); ys.append(y.numpy())
    return np.concatenate(probs), np.concatenate(ys)

def per_class_auc(y, p):
    return {lab: (roc_auc_score(y[:, i], p[:, i]) if 0 < y[:, i].sum() < len(y) else np.nan)
            for i, lab in enumerate(LABELS)}

history, best_auc, bad_epochs = [], -1.0, 0
for epoch in range(1, EPOCHS + 1):
    model.train(); t0, running = time.time(), 0.0
    for step, (x, y) in enumerate(train_dl, 1):
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update(); scheduler.step()
        running += loss.item()
        if step % 200 == 0:
            print(f"  epoch {epoch} step {step}/{len(train_dl)} loss {running / step:.4f}")

    p_val, y_val = predict(val_dl)
    val_auc = np.nanmean(list(per_class_auc(y_val, p_val).values()))
    history.append({"epoch": epoch, "train_loss": running / len(train_dl), "val_macro_auc": val_auc})
    print(f"Epoch {epoch}: train loss {running / len(train_dl):.4f} | val macro-AUC {val_auc:.4f} | {time.time() - t0:.0f}s")

    if val_auc > best_auc:
        best_auc, bad_epochs = val_auc, 0
        torch.save(model.state_dict(), f"{OUT_DIR}/densenet121_chestxray_best.pt")
        print("  -> saved best model")
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print("Early stopping"); break

hist = pd.DataFrame(history); hist.to_csv(f"{OUT_DIR}/training_history.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
hist.plot(x="epoch", y="train_loss", ax=ax[0], marker="o", title="Train loss", legend=False)
hist.plot(x="epoch", y="val_macro_auc", ax=ax[1], marker="o", title="Validation macro-AUC", legend=False)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/training_curves.png", dpi=120); plt.show()

## 8. Evaluate on held-out test patients

In [ ]:
model.load_state_dict(torch.load(f"{OUT_DIR}/densenet121_chestxray_best.pt", map_location=DEVICE))
p_test, y_test = predict(test_dl)
aucs = per_class_auc(y_test, p_test)
macro_auc = float(np.nanmean(list(aucs.values())))

results = (pd.DataFrame({"AUC": aucs, "test_positives": y_test.sum(0).astype(int)})
             .sort_values("AUC", ascending=False))
results.loc["MACRO AVERAGE", "AUC"] = macro_auc
results["AUC"] = results["AUC"].round(3)
print(results.to_string())
results.to_csv(f"{OUT_DIR}/test_auc_per_class.csv")

with open(f"{OUT_DIR}/metrics.json", "w") as f:
    json.dump({"macro_auc": round(macro_auc, 4), "per_class_auc": {k: round(float(v), 4) for k, v in aucs.items()},
               "n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
               "epochs_run": len(history), "best_val_macro_auc": round(float(best_auc), 4)}, f, indent=2)

plt.figure(figsize=(8, 7))
for i, lab in enumerate(LABELS):
    if 0 < y_test[:, i].sum() < len(y_test):
        fpr, tpr, _ = roc_curve(y_test[:, i], p_test[:, i])
        plt.plot(fpr, tpr, lw=1.2, label=f"{lab} ({aucs[lab]:.2f})")
plt.plot([0, 1], [0, 1], "k--", lw=0.8)
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title(f"Test ROC curves — macro AUC {macro_auc:.3f}")
plt.legend(fontsize=8, loc="lower right"); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/roc_curves.png", dpi=120); plt.show()

## 9. Grad-CAM — where is the model looking?

Gradients of a class score with respect to the last DenseNet feature map are averaged into channel weights; the weighted feature map gives a heatmap of the regions that drove the prediction.

In [ ]:
def grad_cam(x, class_idx):
    model.eval(); store = {}
    def hook(_, __, out):
        store["act"] = out
        out.register_hook(lambda g: store.__setitem__("grad", g))
    h = model.features.register_forward_hook(hook)
    logits = model(x)
    model.zero_grad(); logits[0, class_idx].backward()
    h.remove()
    act, grad = store["act"][0], store["grad"][0]               # (C, h, w)
    cam = F.relu((grad.mean(dim=(1, 2))[:, None, None] * act).sum(0))
    cam = F.interpolate(cam[None, None], size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam.detach().cpu().numpy(), torch.sigmoid(logits[0]).detach().cpu().numpy()

# pick test images where the model confidently and correctly finds a condition
examples = []
for i in np.argsort(-(p_test * y_test).max(1)):
    c = int(np.argmax(p_test[i] * y_test[i]))
    if y_test[i, c] == 1 and LABELS[c] not in [LABELS[e[1]] for e in examples]:
        examples.append((i, c))
    if len(examples) == 6:
        break

assert examples, "no positive test examples found (expected only in a very small smoke test)"
fig, axes = plt.subplots(2, len(examples), figsize=(3.2 * len(examples), 6.6))
axes = np.array(axes).reshape(2, -1)
for col, (i, c) in enumerate(examples):
    x = eval_tf(torch.from_numpy(X_test[i]).float().div(255.).unsqueeze(0).repeat(3, 1, 1))[None].to(DEVICE)
    cam, probs = grad_cam(x, c)
    img = cv2.resize(X_test[i], (IMG_SIZE, IMG_SIZE))
    axes[0, col].imshow(img, cmap="gray"); axes[0, col].set_title(f"True: {LABELS[c]}", fontsize=9)
    axes[1, col].imshow(img, cmap="gray"); axes[1, col].imshow(cam, cmap="jet", alpha=0.4)
    axes[1, col].set_title(f"P({LABELS[c]}) = {probs[c]:.2f}", fontsize=9)
for a in axes.ravel(): a.axis("off")
plt.suptitle("Grad-CAM on held-out test X-rays", y=1.0); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/gradcam_examples.png", dpi=120, bbox_inches="tight"); plt.show()

## 10. Summary for the README

In [ ]:
print(f"Trained on {len(train_df):,} images ({train_df.patient_id.nunique():,} patients), "
      f"tested on {len(test_df):,} held-out images ({test_df.patient_id.nunique():,} patients).")
print(f"Test macro ROC-AUC: {macro_auc:.3f}")
print("Top classes:", ", ".join(f"{k} {v:.2f}" for k, v in sorted(aucs.items(), key=lambda kv: -kv[1])[:4]))
print("\nDownload everything in", OUT_DIR, ":", sorted(os.listdir(OUT_DIR)))